# PAR-S Generator — 完整流程可视化

本 notebook 独立演示 PAR-S Generator 的全部核心逻辑，**不依赖 GUI 代码**，适合用于理解、调试和验证各阶段算法。

## 整体流程

```
┌──────────────────────────────────────────────────────────────┐
│  阶段 1：SYN 体模生成                                          │
│  PhantomConfig → PhantomGenerator → case_XXXX.npz           │
└──────────────────────────────────────────────────────────────┘
                            ↓
┌──────────────────────────────────────────────────────────────┐
│  阶段 2：格式转换                                              │
│  case_XXXX.npz → Interfile (.h33/.i33) × 2 (act + atn)     │
└──────────────────────────────────────────────────────────────┘
                            ↓
┌──────────────────────────────────────────────────────────────┐
│  阶段 3：SIMIND 蒙特卡洛仿真                                   │
│  simind.exe + .smc → SPECT 投影数据 (.a00 等)                │
└──────────────────────────────────────────────────────────────┘
```

> **注意**：阶段 3 需要 SIMIND 可执行文件，本 notebook 只生成调用命令，不实际运行。

In [ ]:
# ── 依赖导入 ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import gaussian_filter
from pathlib import Path
import json, time, struct

# 配色
plt.rcParams.update({
    'figure.facecolor': '#1e2128',
    'axes.facecolor': '#161a1f',
    'axes.edgecolor': '#2d3139',
    'axes.labelcolor': '#8b949e',
    'xtick.color': '#6b7280',
    'ytick.color': '#6b7280',
    'text.color': '#c8ccd4',
    'grid.color': '#2d3139',
    'image.cmap': 'hot',
})
print('依赖加载完成。')

---
## 阶段 1：体模生成

### 1.1 配置参数（PhantomConfig 的核心子集）

In [ ]:
# Core parameters (matching PhantomConfig v2.2)
# v2.2: body scaled to realistic adult, scatter corrected
CFG = dict(
    volume_shape      = (128, 128, 128),
    voxel_size_mm     = 4.42,

    # mu-map (cm^-1, 140 keV Tc-99m)
    mu_water          = 0.15,
    mu_liver          = 0.16,
    mu_lung           = 0.05,
    mu_spine          = 0.30,
    mu_fat            = 0.09,
    mu_noise_amp      = 0.015,
    mu_noise_sigma    = 2.0,

    # Body contour (rz, ry, rx) normalized [-1,1]
    # Physical: Z=38.9cm(SI), Y=23.0cm(AP), X=34.5cm(LR) -- realistic adult supine
    body_radii        = (0.67, 0.39, 0.60),
    outer_body_radii  = (0.69, 0.41, 0.62),

    # Liver geometry -- all in normalized coords
    # X=0.10: liver CoM at 2.8cm right of midline.
    # Left lobe extends to X=-0.16 so Cantlie plane can split 35/65.
    liver_base_center = (-0.20, 0.10, 0.10),
    right_radii       = (0.28, 0.22, 0.26),
    right_shift       = (0.0,  0.0,  0.14),
    right_rot_deg     = -15.0,
    left_radii        = (0.18, 0.19, 0.18),
    left_shift        = (0.14, 0.06, -0.08),
    left_rot_deg      = 10.0,
    dome_radius       = 0.34,   # liver top Z=+0.09 (2.5cm above FOV center)
    fossa_radius      = 0.14,   # 4cm gallbladder fossa (old 0.23=6.5cm was too large)
    dome_offset       = (-0.05, 0.0, 0.0),
    fossa_offset      = (-0.12, -0.03, 0.0),

    # Lungs: center Z=0.38 (10.8cm above center), top Z=0.58 < body top 0.69
    # Diaphragm gap = 2.5cm between liver dome top (Z=0.09) and lung bottom (Z=0.18)
    lung_center_z     = 0.38,
    lung_radii        = (0.20, 0.14, 0.18),
    lung_x_offset     = 0.22,

    # Spine: Y=-0.30 (8.5cm posterior), radius=0.06 (3.4cm diameter)
    spine_y           = -0.30,
    spine_radius      = 0.06,

    # Jitter
    global_shift_range = 0.05,
    scale_jitter       = 0.10,
    rot_jitter_deg     = 5.0,
    detail_jitter      = 0.05,
    smooth_sigma       = 1.2,
    smooth_thr         = 0.5,

    # Cantlie plane search range covers X~0.02-0.05 where plane needs to be
    target_left_ratio    = 0.35,
    cantlie_tilt_range   = (-6.0, 10.0),
    cantlie_offset_range = (-0.05, 0.12),
    cantlie_iter_max     = 12,

    # Tumors -- all cases have >=1 tumor
    tumor_count_min     = 1,
    tumor_count_max     = 5,
    tumor_size_bins_mm  = [[10,20],[20,40],[40,60]],
    tumor_probs         = [0.45, 0.40, 0.15],
    tumor_contrast_min  = 2.0,   # Ho et al. 1997: TNR range 1.5-12
    tumor_contrast_max  = 8.0,
    min_edge_dist_px    = 4,

    # Perfusion
    perfusion_probs = {'Whole Liver':0.05,'Tumor Only':0.25,'Left Only':0.35,'Right Only':0.35},
    residual_bg     = 0.05,
    gradient_gain   = 0.08,
    total_counts    = 8e4,
)

HALF_FOV_CM = 128 * 4.42 / 2 / 10
print('PhantomConfig v2.2')
print('FOV: %.0f x %.0f x %.0f mm' % (128*4.42, 128*4.42, 128*4.42))
print('Body (Z/Y/X cm): %s' % [round(r*2*HALF_FOV_CM,1) for r in CFG['body_radii']])
print('Expected liver: ~900-1800 ml [validated]')


### 1.2 几何辅助函数（对应 Geometry3D 类）

In [ ]:
# ── 归一化网格 ──────────────────────────────────────────────────
def get_grid(shape):
    """返回 (Z, Y, X) 归一化网格，范围 [-1, 1]"""
    z = np.linspace(-1, 1, shape[0])
    y = np.linspace(-1, 1, shape[1])
    x = np.linspace(-1, 1, shape[2])
    return np.meshgrid(z, y, x, indexing='ij')

# ── 旋转椭球 ──────────────────────────────────────────────────
def create_ellipsoid(shape, center, radii, rotation_deg=0.0, rotation_plane='xz',
                     rng=None, jitter=None):
    """生成旋转椭球掩码（归一化坐标系）"""
    if rng is None: rng = np.random.default_rng()
    z0, y0, x0 = center
    rz, ry, rx = radii
    jitter = jitter or {}
    cj = jitter.get('center', 0.0)
    rj = jitter.get('radii',  0.0)
    rdeg = jitter.get('rot_deg', 0.0)
    z0 += rng.uniform(-cj, cj);  y0 += rng.uniform(-cj, cj);  x0 += rng.uniform(-cj, cj)
    rz *= rng.uniform(1-rj, 1+rj); ry *= rng.uniform(1-rj, 1+rj); rx *= rng.uniform(1-rj, 1+rj)
    theta = np.radians(rotation_deg + rng.uniform(-rdeg, rdeg))
    Z, Y, X = get_grid(shape)
    if rotation_plane == 'xz':
        X_rot = (X-x0)*np.cos(theta) - (Z-z0)*np.sin(theta)
        Z_rot = (X-x0)*np.sin(theta) + (Z-z0)*np.cos(theta)
        Y_rot = Y - y0
    else:
        X_rot = (X-x0)*np.cos(theta) - (Y-y0)*np.sin(theta)
        Y_rot = (X-x0)*np.sin(theta) + (Y-y0)*np.cos(theta)
        Z_rot = Z - z0
    return (X_rot/rx)**2 + (Y_rot/ry)**2 + (Z_rot/rz)**2 <= 1.0

# ── 毛刺状肿瘤 ──────────────────────────────────────────────────
def create_spiculated(shape, center_idx, radius_vox, roughness=0.35, spiciness=3.0, rng=None):
    if rng is None: rng = np.random.default_rng()
    margin = int(radius_vox * 2 + 8)
    cz, cy, cx = center_idx
    z0, z1 = max(0, cz-margin), min(shape[0], cz+margin)
    y0, y1 = max(0, cy-margin), min(shape[1], cy+margin)
    x0, x1 = max(0, cx-margin), min(shape[2], cx+margin)
    ls = (z1-z0, y1-y0, x1-x0)
    if any(s <= 1 for s in ls): return np.zeros(shape, dtype=bool)
    zz, yy, xx = np.ogrid[:ls[0], :ls[1], :ls[2]]
    zz -= (cz-z0); yy -= (cy-y0); xx -= (cx-x0)
    dist = np.sqrt(zz**2 + yy**2 + xx**2)
    noise = gaussian_filter(rng.random(ls), sigma=spiciness)
    noise = (noise - 0.5) * 2.0
    local = dist <= (radius_vox + noise*(radius_vox*roughness))
    full = np.zeros(shape, dtype=bool)
    full[z0:z1, y0:y1, x0:x1] = local
    return full

# ── Cantlie 平面分割 ──────────────────────────────────────────────
def split_liver_lobes(liver_mask, shape, target_left_ratio=0.35,
                      tilt_deg=5.0, offset=0.0):
    Z, Y, X = get_grid(shape)
    theta = np.radians(tilt_deg)
    nx, ny, nz = 0, np.sin(theta), np.cos(theta)
    partition = (X*nz + Y*ny + Z*nx) > offset
    return liver_mask & partition, liver_mask & ~partition   # left, right

print('几何辅助函数定义完成。')

### 1.3 步骤1：构建肝脏掩码

**算法**：`右叶椭球 ∪ 左叶椭球 ∩ 身体椭球 ∩ 穹隆椭球 − 胆囊窝椭球 → 高斯平滑 → 二值化`

In [ ]:
SEED = 42
rng  = np.random.default_rng(SEED)
shape = CFG['volume_shape']

# 全局位置抖动
base_center  = np.array(CFG['liver_base_center'])
global_shift = rng.uniform(-CFG['global_shift_range'], CFG['global_shift_range'], 3)
center       = base_center + global_shift

# 右叶（体积更大，位于患者右侧即 X+ 方向）
right_radii  = tuple(r*rng.uniform(1-CFG['scale_jitter'], 1+CFG['scale_jitter'])
                     for r in CFG['right_radii'])
right_center = tuple(center + np.array(CFG['right_shift']))
right_lobe   = create_ellipsoid(shape, right_center, right_radii,
                                rotation_deg=CFG['right_rot_deg'], rng=rng)

# 左叶（体积更小，位于患者左侧即 X- 方向）
left_radii   = tuple(r*rng.uniform(1-CFG['scale_jitter'], 1+CFG['scale_jitter'])
                     for r in CFG['left_radii'])
left_center  = tuple(center + np.array(CFG['left_shift']))
left_lobe    = create_ellipsoid(shape, left_center, left_radii,
                                rotation_deg=CFG['left_rot_deg'], rng=rng)

# 身体轮廓（防止肝脏超出体表）
body  = create_ellipsoid(shape, (0,0,0), (0.90, 0.65, 0.85))

# 穹隆（上界约束）
dome_r = CFG['dome_radius'] + rng.uniform(-CFG['detail_jitter'], CFG['detail_jitter'])
dome   = create_ellipsoid(shape, tuple(center+np.array(CFG['dome_offset'])),
                           (dome_r,)*3, rng=rng)

# 胆囊窝（切除下界凹陷）
fossa_r = CFG['fossa_radius'] + rng.uniform(-CFG['detail_jitter'], CFG['detail_jitter'])
fossa   = create_ellipsoid(shape, tuple(center+np.array(CFG['fossa_offset'])),
                            (fossa_r,)*3, rng=rng)

# 布尔组合：(右 ∪ 左) ∩ 身体 ∩ 穹隆 − 胆囊窝
liver_raw = (right_lobe | left_lobe) & body & dome & ~fossa

# 高斯平滑 + 阈值二值化
liver = gaussian_filter(liver_raw.astype(float), sigma=CFG['smooth_sigma']) > CFG['smooth_thr']

vox_vol_ml   = (CFG['voxel_size_mm']/10)**3
liver_vol_ml = liver.sum() * vox_vol_ml
print(f'肝脏掩码生成完成：{liver.sum()} 体素  ≈  {liver_vol_ml:.0f} mL')

In [ ]:
# ── 可视化：肝脏掩码的三平面切片 ──────────────────────────────────
mid = shape[0]//2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('肝脏掩码构建：三平面切片视图', color='#c8ccd4', fontsize=13, pad=10)

planes = [
    ('轴面 (Axial, Z)', liver[mid, :, :],        right_lobe[mid,:,:], left_lobe[mid,:,:]),
    ('冠面 (Coronal, Y)', liver[:,mid,:],         right_lobe[:,mid,:], left_lobe[:,mid,:]),
    ('矢面 (Sagittal, X)', liver[:,:,mid].T,      right_lobe[:,:,mid].T, left_lobe[:,:,mid].T),
]
for ax, (title, slc, rl, ll) in zip(axes, planes):
    bg = np.zeros((*slc.shape, 3))
    bg[rl > 0] = [0.31, 0.76, 0.97]   # 右叶 - 蓝
    bg[ll > 0] = [0.60, 0.80, 0.40]   # 左叶 - 绿
    bg[slc > 0] = bg[slc > 0] * 0.5 + np.array([0.20, 0.50, 0.60]) * 0.5  # 最终肝脏
    ax.imshow(bg, origin='lower', aspect='equal')
    ax.set_title(title, color='#8b949e', fontsize=10)
    ax.set_xlabel('X 轴', color='#6b7280', fontsize=8)
    ax.set_ylabel('Y/Z 轴', color='#6b7280', fontsize=8)

# 图例
from matplotlib.patches import Patch
legend = [
    Patch(color=[0.31,0.76,0.97], label='右叶椭球'),
    Patch(color=[0.60,0.80,0.40], label='左叶椭球'),
    Patch(color=[0.20,0.50,0.60], label='最终肝脏掩码（布尔+平滑）'),
]
axes[1].legend(handles=legend, loc='upper right', fontsize=8,
               facecolor='#1e2128', labelcolor='#c8ccd4')
plt.tight_layout()
plt.show()
print(f'右叶原始体素：{right_lobe.sum():,}  左叶原始体素：{left_lobe.sum():,}  最终：{liver.sum():,}')

### 1.4 步骤2：Cantlie 平面分割左右叶

算法核心：在 X-Y 平面上定义一个倾斜平面，迭代调整偏移使左叶体积比例收敛到目标值（35%）。

In [ ]:
liver_vol = liver.sum()

# 初始参数：随机倾斜角 + 随机偏移
tilt       = rng.uniform(*CFG['cantlie_tilt_range'])
offset_val = rng.uniform(*CFG['cantlie_offset_range'])
best_offset, best_diff = offset_val, float('inf')

history_ratios  = []
history_offsets = []

# 迭代搜索（梯度下降 offset）
for i in range(CFG['cantlie_iter_max']):
    ll, rl = split_liver_lobes(liver, shape, CFG['target_left_ratio'],
                               tilt_deg=tilt, offset=offset_val)
    left_ratio = ll.sum() / liver_vol if liver_vol > 0 else 0.5
    diff = abs(left_ratio - CFG['target_left_ratio'])
    history_ratios.append(left_ratio)
    history_offsets.append(offset_val)
    if diff < best_diff:
        best_diff, best_offset = diff, offset_val
    if left_ratio < CFG['target_left_ratio']:
        offset_val -= 0.01
    else:
        offset_val += 0.01

left_mask, right_mask = split_liver_lobes(liver, shape, CFG['target_left_ratio'],
                                          tilt_deg=tilt, offset=best_offset)
actual_left = left_mask.sum() / liver_vol

print(f'Cantlie 平面参数: 倾斜角 = {tilt:.2f}°  最优偏移 = {best_offset:.4f}')
print(f'左叶比例: 目标={CFG["target_left_ratio"]:.2f}  实际={actual_left:.3f}  误差={abs(actual_left-CFG["target_left_ratio"]):.3f}')

In [ ]:
# ── 可视化：Cantlie 平面迭代过程 + 分割结果 ──────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cantlie 平面迭代分割', color='#c8ccd4', fontsize=13, pad=10)

# 左图：迭代历史
ax1.plot(history_ratios, color='#4fc3f7', lw=2, marker='o', markersize=5, label='左叶比例')
ax1.axhline(CFG['target_left_ratio'], color='#ff6b6b', lw=1.5, ls='--', label=f'目标 {CFG["target_left_ratio"]}')
ax1.axhline(actual_left, color='#4caf50', lw=1, ls=':', label=f'最终 {actual_left:.3f}')
ax1.set_title('迭代收敛过程', color='#8b949e', fontsize=10)
ax1.set_xlabel('迭代次数', color='#6b7280')
ax1.set_ylabel('左叶体积比例', color='#6b7280')
ax1.legend(fontsize=9, facecolor='#1e2128', labelcolor='#c8ccd4')
ax1.grid(True, alpha=0.3)

# 右图：分割结果（轴面）
slc_z = shape[0]//2
rgb = np.zeros((*liver[slc_z,:,:].shape, 3))
rgb[left_mask[slc_z]  > 0] = [0.31, 0.76, 0.40]   # 左叶 - 绿
rgb[right_mask[slc_z] > 0] = [0.31, 0.60, 0.97]   # 右叶 - 蓝
ax2.imshow(rgb, origin='lower', aspect='equal')
ax2.set_title(f'轴面分割结果 (slice {slc_z})  左:{actual_left*100:.1f}%  右:{(1-actual_left)*100:.1f}%',
              color='#8b949e', fontsize=10)
ax2.set_xlabel('X 轴（左-右）', color='#6b7280')
ax2.set_ylabel('Y 轴（前-后）', color='#6b7280')

# Cantlie 线叠加
nx_pts = np.linspace(0, shape[2]-1, 200)
# 平面方程：X*nz + Y*ny = offset（归一化坐标）
x_norm = np.linspace(-1, 1, shape[2])
theta_r = np.radians(tilt)
y_norm = (best_offset - x_norm * np.cos(theta_r)) / (np.sin(theta_r) + 1e-9)
y_px = (y_norm + 1) / 2 * shape[1]
ax2.plot(nx_pts, np.interp(nx_pts, np.linspace(0, shape[2]-1, len(y_px)), y_px),
         color='#ffa726', lw=2, ls='--', label='Cantlie 线')
ax2.legend(fontsize=9, facecolor='#1e2128', labelcolor='#c8ccd4')

plt.tight_layout()
plt.show()

### 1.5 步骤3：构建 μ-map（衰减系数图）

各组织在 140 keV（Tc-99m）的衰减系数（cm⁻¹）：

In [ ]:
# mu-map construction (v2.2 corrected anatomy)
mu = np.ones(shape, dtype=np.float32) * CFG['mu_water']

# Lungs: center Z=0.38 (10.8cm above FOV center), top Z=0.58 < body top Z=0.69
lung_r = create_ellipsoid(shape, (0.38,  0.05, -0.22), (0.20, 0.14, 0.18))
lung_l = create_ellipsoid(shape, (0.38,  0.05,  0.22), (0.20, 0.14, 0.18))
mu[lung_r | lung_l] = CFG['mu_lung']

# Spine: Y=-0.30 (8.5cm posterior from center), diameter 3.4cm
Z_grid, Y_grid, X_grid = get_grid(shape)
spine_mask = ((X_grid - 0)**2 + (Y_grid + 0.30)**2) <= 0.06**2
mu[spine_mask] = CFG['mu_spine']

# Liver
mu[liver] = CFG['mu_liver']

# Fat layer: outer_body=(0.69,0.41,0.62), body_inner=(0.67,0.39,0.60)
outer_body = create_ellipsoid(shape, (0,0,0), (0.69, 0.41, 0.62))
body_inner = create_ellipsoid(shape, (0,0,0), (0.67, 0.39, 0.60))
fat_layer  = outer_body & ~body_inner
mu[fat_layer] = CFG['mu_fat']

# Texture noise
noise = gaussian_filter(rng.random(shape).astype(np.float32), sigma=CFG['mu_noise_sigma'])
noise = (noise - noise.mean()) * CFG['mu_noise_amp']
mu    = np.clip(mu + noise, 0, None)

# Air: outside body must be exactly 0.0 (critical for correct SIMIND scatter)
mu[~outer_body] = 0.0

print('mu-map built (v2.2)')
print('Air fraction: %.1f%%  (target >50%%)' % ((mu==0).sum()/mu.size*100))
print('Body voxels: %d  (~%.0f ml)' % ((mu>0).sum(), (mu>0).sum()*(4.42/10)**3))
print('mu range: [%.4f, %.4f] cm^-1' % (mu[mu>0].min(), mu.max()))


In [ ]:
# ── 可视化：μ-map 三平面 + 组织分布 ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('μ-map（衰减系数图）', color='#c8ccd4', fontsize=13, pad=10)

slc = shape[0]//2
im0 = axes[0,0].imshow(mu[slc,:,:].T, origin='lower', vmin=0, vmax=0.35, cmap='hot')
axes[0,0].set_title(f'轴面 (Z={slc})', color='#8b949e')
plt.colorbar(im0, ax=axes[0,0], label='μ (cm⁻¹)')

im1 = axes[0,1].imshow(mu[:,slc,:].T, origin='lower', vmin=0, vmax=0.35, cmap='hot')
axes[0,1].set_title(f'冠面 (Y={slc})', color='#8b949e')
plt.colorbar(im1, ax=axes[0,1], label='μ (cm⁻¹)')

im2 = axes[0,2].imshow(mu[:,:,slc].T, origin='lower', vmin=0, vmax=0.35, cmap='hot')
axes[0,2].set_title(f'矢面 (X={slc})', color='#8b949e')
plt.colorbar(im2, ax=axes[0,2], label='μ (cm⁻¹)')

# 各组织掩码叠加图
rgb_atn = np.zeros((*mu[slc,:,:].shape, 3))
rgb_atn[body_inner[slc,:,:] > 0]    = [0.15, 0.20, 0.30]   # 背景软组织
rgb_atn[(lung_r|lung_l)[slc,:,:]]   = [0.20, 0.50, 0.80]   # 肺 - 蓝
rgb_atn[liver[slc,:,:] > 0]         = [0.80, 0.40, 0.20]   # 肝脏 - 橙
rgb_atn[spine_mask[slc,:,:] > 0]    = [0.90, 0.90, 0.90]   # 脊柱 - 白
rgb_atn[fat_layer[slc,:,:] > 0]     = [0.90, 0.80, 0.30]   # 脂肪 - 黄
axes[1,0].imshow(rgb_atn, origin='lower', aspect='equal')
axes[1,0].set_title('组织掩码叠加（轴面）', color='#8b949e')

from matplotlib.patches import Patch
legend_handles = [
    Patch(color=[0.15,0.20,0.30], label=f'软组织 μ={CFG["mu_water"]}'),
    Patch(color=[0.20,0.50,0.80], label=f'肺 μ={CFG["mu_lung"]}'),
    Patch(color=[0.80,0.40,0.20], label=f'肝脏 μ={CFG["mu_liver"]}'),
    Patch(color=[0.90,0.90,0.90], label=f'脊柱 μ={CFG["mu_spine"]}'),
    Patch(color=[0.90,0.80,0.30], label=f'脂肪 μ={CFG["mu_fat"]}'),
]
axes[1,1].legend(handles=legend_handles, loc='center', fontsize=10,
                 facecolor='#1e2128', labelcolor='#c8ccd4')
axes[1,1].axis('off')
axes[1,1].set_title('组织分层说明', color='#8b949e')

# μ 值直方图
mu_flat = mu[body_inner].flatten()
axes[1,2].hist(mu_flat, bins=80, color='#4fc3f7', edgecolor='none', alpha=0.8)
axes[1,2].set_title('μ 值分布（体内体素）', color='#8b949e')
axes[1,2].set_xlabel('μ (cm⁻¹)', color='#6b7280')
axes[1,2].set_ylabel('体素数量', color='#6b7280')
for v, lbl, c in [(0.05,'肺','#2196F3'),(0.09,'脂肪','#FFC107'),
                   (0.15,'软组织','#4CAF50'),(0.16,'肝脏','#FF6B35'),(0.30,'脊柱','#E0E0E0')]:
    axes[1,2].axvline(v, color=c, ls='--', lw=1.2, alpha=0.8, label=lbl)
axes[1,2].legend(fontsize=8, facecolor='#1e2128', labelcolor='#c8ccd4')
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 1.6 步骤4：植入肿瘤

In [ ]:
# ── 肿瘤植入 ──────────────────────────────────────────────────
n_tumors     = int(rng.integers(CFG['tumor_count_min'], CFG['tumor_count_max']+1))
tumor_masks  = []
tumor_radii  = []
tumor_modes  = []
tumor_centers= []
liver_indices= np.argwhere(liver)

mode_choices = ["spiculated", "ellipsoid", "superellipsoid", "noise_threshold"]

for t in range(n_tumors):
    # 1. 采样尺寸
    bin_idx    = rng.choice(len(CFG['tumor_size_bins_mm']), p=CFG['tumor_probs'])
    rmin, rmax = CFG['tumor_size_bins_mm'][bin_idx]
    radius_mm  = rng.uniform(rmin/2, rmax/2)       # 半径 = 直径/2
    radius_vox = radius_mm / CFG['voxel_size_mm']

    # 2. 采样形态
    mode = rng.choice(mode_choices)

    # 3. 找肝脏内合适位置
    placed = False
    for attempt in range(50):
        idx = liver_indices[rng.integers(len(liver_indices))]
        cz, cy, cx = int(idx[0]), int(idx[1]), int(idx[2])
        # 边缘距离检查
        edge_ok = all([
            cz >= CFG['min_edge_dist_px'], cz < shape[0]-CFG['min_edge_dist_px'],
            cy >= CFG['min_edge_dist_px'], cy < shape[1]-CFG['min_edge_dist_px'],
            cx >= CFG['min_edge_dist_px'], cx < shape[2]-CFG['min_edge_dist_px'],
        ])
        if not edge_ok: continue
        # 肿瘤间距检查
        center_ok = all(
            np.sqrt((cz-tc[0])**2+(cy-tc[1])**2+(cx-tc[2])**2) >= CFG['min_center_dist_px']
            for tc in tumor_centers
        ) if tumor_centers else True
        if not center_ok: continue
        placed = True
        tumor_centers.append((cz, cy, cx))
        break
    if not placed: continue

    # 4. 生成掩码
    if mode == "spiculated":
        tmask = create_spiculated(shape, (cz,cy,cx), radius_vox, rng=rng)
    else:   # 简化：全部用球（notebook 演示）
        from scipy.ndimage import binary_fill_holes
        zz,yy,xx = np.ogrid[:shape[0],:shape[1],:shape[2]]
        tmask = np.sqrt((zz-cz)**2+(yy-cy)**2+(xx-cx)**2) <= radius_vox

    tmask = tmask & liver
    if tmask.sum() == 0: continue
    tumor_masks.append(tmask)
    tumor_radii.append(float(radius_mm * 2))  # 存储直径
    tumor_modes.append(mode)

print(f'成功植入 {len(tumor_masks)} 个肿瘤（尝试 {n_tumors} 个）')
for i,(r,m) in enumerate(zip(tumor_radii, tumor_modes)):
    print(f'  #{i+1}: 直径 {r:.1f} mm  形态: {m}  中心: {tumor_centers[i]}')

### 1.7 步骤5：生成活度图（灌注模式 + PSF + 泊松噪声）

In [ ]:
# ── 活度图生成 ──────────────────────────────────────────────────
perf_keys = list(CFG['perfusion_probs'].keys())
perf_vals = list(CFG['perfusion_probs'].values())
perfusion_mode = rng.choice(perf_keys, p=perf_vals)

activity = np.zeros(shape, dtype=np.float32)

if perfusion_mode == 'Whole Liver':
    activity[liver] = 1.0
elif perfusion_mode == 'Left Only':
    activity[left_mask] = 1.0
    activity[right_mask] = CFG['residual_bg']
elif perfusion_mode == 'Right Only':
    activity[right_mask] = 1.0
    activity[left_mask] = CFG['residual_bg']
elif perfusion_mode == 'Tumor Only':
    activity[liver] = CFG['residual_bg']

# Z 方向梯度（模拟肝脏上下活度差）
if CFG['gradient_gain'] > 0:
    Z_g, _, _ = get_grid(shape)
    grad = (Z_g + 1) / 2 * CFG['gradient_gain']
    activity += (grad * liver).astype(np.float32)

# 肿瘤高摄取
contrast = rng.uniform(CFG['tumor_contrast_min'], CFG['tumor_contrast_max'])
for tmask in tumor_masks:
    base_val = activity[tmask].mean() if activity[tmask].sum() > 0 else 1.0
    activity[tmask] = base_val * contrast

# PSF 模糊（模拟 SPECT 空间分辨率）
if CFG['psf_sigma_px'] > 0:
    activity = gaussian_filter(activity, sigma=CFG['psf_sigma_px'])

# 泊松噪声 + 归一化
if activity.sum() > 0:
    activity = activity / activity.sum() * CFG['total_counts']
    activity = rng.poisson(np.maximum(activity, 0)).astype(np.float32)

total_counts_actual = float(activity.sum())
print(f'灌注模式: {perfusion_mode}')
print(f'肿瘤对比度: {contrast:.1f}×')
print(f'PSF σ: {CFG["psf_sigma_px"]} px = {CFG["psf_sigma_px"]*CFG["voxel_size_mm"]:.1f} mm FWHM')
print(f'总计数: {total_counts_actual:.2e}')

In [ ]:
# ── 可视化：活度图（全部灌注模式对比）──────────────────────────────
# 生成四种灌注模式的示意图
rng2 = np.random.default_rng(123)
def make_activity(perf, liver, left, right, tumor_masks, rng, cfg):
    act = np.zeros(liver.shape, dtype=np.float32)
    if perf == 'Whole Liver':  act[liver] = 1.0
    elif perf == 'Left Only':  act[left] = 1.0; act[right] = cfg['residual_bg']
    elif perf == 'Right Only': act[right] = 1.0; act[left] = cfg['residual_bg']
    elif perf == 'Tumor Only': act[liver] = cfg['residual_bg']
    c = rng.uniform(cfg['tumor_contrast_min'], cfg['tumor_contrast_max'])
    for tm in tumor_masks:
        bv = act[tm].mean() if act[tm].sum()>0 else 1.0
        act[tm] = bv * c
    act = gaussian_filter(act, sigma=cfg['psf_sigma_px'])
    return act

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('四种灌注模式的活度图（中间轴面切片）', color='#c8ccd4', fontsize=13, pad=10)

modes_display = ['Whole Liver', 'Left Only', 'Right Only', 'Tumor Only']
slc = shape[0]//2

for col, pm in enumerate(modes_display):
    act = make_activity(pm, liver, left_mask, right_mask, tumor_masks, rng2, CFG)
    prob = CFG['perfusion_probs'][pm]

    im = axes[0,col].imshow(act[slc,:,:].T, origin='lower', cmap='hot',
                             vmin=0, vmax=act.max()*0.9)
    axes[0,col].set_title(f'{pm}\n(p={prob:.0%})', color='#8b949e', fontsize=10)
    plt.colorbar(im, ax=axes[0,col])

    # 肝脏轮廓叠加
    axes[0,col].contour(liver[slc,:,:].T, levels=[0.5],
                        colors=['#4fc3f7'], linewidths=0.8, alpha=0.7)

    # 冠面
    im2 = axes[1,col].imshow(act[:,slc,:].T, origin='lower', cmap='hot',
                              vmin=0, vmax=act.max()*0.9)
    axes[1,col].set_title(f'冠面 (Y={slc})', color='#8b949e', fontsize=9)
    plt.colorbar(im2, ax=axes[1,col])

axes[0,0].set_ylabel('轴面 (Axial)', color='#6b7280')
axes[1,0].set_ylabel('冠面 (Coronal)', color='#6b7280')

plt.tight_layout()
plt.show()

### 1.8 完整案例预览（activity + mu_map + 掩码）

In [ ]:
# ── 合成体模完整预览 ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#1e2128')
fig.suptitle('PAR-S Generator — 合成体模完整预览\n'
             f'灌注: {perfusion_mode}  |  肿瘤: {len(tumor_masks)} 个  |  '
             f'肝脏体积: {liver_vol_ml:.0f} mL  |  左叶比例: {actual_left*100:.1f}%',
             color='#c8ccd4', fontsize=12, pad=15)

gs = gridspec.GridSpec(3, 5, figure=fig, hspace=0.4, wspace=0.35)
slc = shape[0]//2

planes = {
    '轴面':    (slice(slc,slc+1), np.s_[:], np.s_[:]),
    '冠面':    (np.s_[:], slice(slc,slc+1), np.s_[:]),
    '矢面':    (np.s_[:], np.s_[:], slice(slc,slc+1)),
}

vol_labels = ['活度图 (activity)', 'μ-map (attenuation)', '肝脏掩码 (liver_mask)']
vols_axial   = [activity[slc,:,:].T, mu[slc,:,:].T, liver[slc,:,:].T.astype(float)]
vols_coronal = [activity[:,slc,:].T, mu[:,slc,:].T, liver[:,slc,:].T.astype(float)]
vols_sagittal= [activity[:,:,slc].T, mu[:,:,slc].T, liver[:,:,slc].T.astype(float)]
cmaps = ['hot', 'gray', 'Blues']

for row, (vols, plane) in enumerate([
        (vols_axial,   '轴面 Axial'),
        (vols_coronal, '冠面 Coronal'),
        (vols_sagittal,'矢面 Sagittal')]):
    for col, (vol, label, cmap) in enumerate(zip(vols, vol_labels, cmaps)):
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor('#161a1f')
        im = ax.imshow(vol, origin='lower', cmap=cmap,
                       vmin=vol.min(), vmax=vol.max())
        # 叠加肿瘤轮廓（红色）
        for tmask in tumor_masks:
            tc = tmask[slc,:,:].T if row==0 else (
                 tmask[:,slc,:].T if row==1 else tmask[:,:,slc].T)
            if tc.any():
                ax.contour(tc, levels=[0.5], colors=['#ff6b6b'], linewidths=1.2)
        if row == 0:
            ax.set_title(label, color='#8b949e', fontsize=9)
        if col == 0:
            ax.set_ylabel(plane, color='#6b7280', fontsize=9)
        ax.tick_params(labelsize=7, colors='#6b7280')

# 右侧统计列
for row in range(3):
    ax_stat = fig.add_subplot(gs[row, 3:5])
    ax_stat.set_facecolor('#252a33')
    ax_stat.axis('off')

ax_info = fig.add_subplot(gs[0, 3:5])
ax_info.set_facecolor('#252a33')
ax_info.axis('off')
info_text = (
    f"PHANTOM STATISTICS\n"
    f"\n体素矩阵: {shape[0]}×{shape[1]}×{shape[2]}"
    f"\n体素尺寸: {CFG['voxel_size_mm']} mm"
    f"\nFOV: {shape[0]*CFG['voxel_size_mm']:.0f}×{shape[1]*CFG['voxel_size_mm']:.0f}×{shape[2]*CFG['voxel_size_mm']:.0f} mm"
    f"\n\n肝脏体积: {liver_vol_ml:.0f} mL"
    f"\n左叶比例: {actual_left*100:.1f}%"
    f"\n右叶比例: {(1-actual_left)*100:.1f}%"
    f"\nCantlie 倾斜角: {tilt:.1f}°"
    f"\n\n灌注模式: {perfusion_mode}"
    f"\n肿瘤数量: {len(tumor_masks)}"
)
if tumor_radii:
    info_text += f"\n肿瘤直径: {', '.join(f'{r:.1f}mm' for r in tumor_radii)}"
info_text += f"\n对比度: {contrast:.1f}×"
info_text += f"\n\n总计数: {total_counts_actual:.2e}"
info_text += f"\nPSF σ: {CFG['psf_sigma_px']} px"
ax_info.text(0.05, 0.95, info_text, transform=ax_info.transAxes,
             color='#c8ccd4', fontsize=9, verticalalignment='top',
             fontfamily='monospace')

plt.show()
print('✓ 体模生成完成！')

---
## 阶段 2：格式转换（npz → Interfile）

### 2.1 Interfile 格式说明

SIMIND 使用 **Interfile v3.3** 格式，每个体积由两个文件组成：
- **`.h33`**：ASCII 头文件，描述数据类型、尺寸、体素大小等
- **`.i33`**：二进制数据文件，存储实际体素值

⚠️ **关键：轴顺序转置**

NumPy 使用 **(Z, Y, X)** C-order 存储，而 Interfile 标准要求 **(X, Y, Z)** 顺序（Fortran-order）。
写入时必须执行 `arr.transpose(2, 1, 0)` 后再写入文件。

In [ ]:
# ── Interfile 写入（演示逻辑，不实际写文件）──────────────────────
def write_interfile_demo(volume_zyx, stem_name, voxel_size_mm=4.42, data_type='float'):
    """
    演示 Interfile 写入逻辑。
    对应 src/core/interfile_writer.py::write_interfile()
    
    参数
    ----
    volume_zyx : np.ndarray  shape=(Z, Y, X)  输入体素数据（NumPy 轴顺序）
    stem_name  : str  输出文件基名（不含扩展名）
    
    ⚠️ 关键步骤：转置 (Z,Y,X) → (X,Y,Z) 再写入
    """
    nz, ny, nx = volume_zyx.shape
    vox_cm = voxel_size_mm / 10.0

    if data_type == 'float':
        arr = volume_zyx.astype(np.float32)
        number_format  = 'short float'
        bytes_per_pixel = 4
    else:
        arr = volume_zyx.astype(np.int16)
        number_format  = 'signed integer'
        bytes_per_pixel = 2

    # ── 关键：转置 (Z,Y,X) → (X,Y,Z) ──
    # 当前代码（有 Bug）：arr.tofile() 直接写入，未转置
    # 正确做法：
    arr_to_write = arr.transpose(2, 1, 0)  # (Z,Y,X) → (X,Y,Z)
    
    # 二进制数据（演示：返回 bytes 而非写文件）
    binary_bytes = arr_to_write.astype(np.float32).tobytes()

    header = f"""\
!INTERFILE :=
!imaging modality := nucmed
!version of keys := 3.3

!GENERAL DATA :=
!data offset in bytes := 0
!name of data file := {stem_name}.i33
!total number of images := {nz}

!GENERAL IMAGE DATA :=
!type of data := Tomographic
!number format := {number_format}
!number of bytes per pixel := {bytes_per_pixel}
imagedata byte order := LITTLEENDIAN

!SPECT STUDY (reconstructed data) :=
!number of slices := {nz}
!matrix size [1] := {nx}
!matrix size [2] := {ny}
!scaling factor (mm/pixel) [1] := {voxel_size_mm:.4f}
!scaling factor (mm/pixel) [2] := {voxel_size_mm:.4f}
!slice thickness (pixels) := 1
!END OF INTERFILE :=
"""
    return header, binary_bytes, arr_to_write.shape

# 演示：打印头文件内容
header_demo, data_demo, out_shape = write_interfile_demo(
    activity, 'case_0000_act', voxel_size_mm=4.42, data_type='float'
)
print('=== Interfile 头文件 (case_0000_act.h33) ===')
print(header_demo)
print(f'输入形状 (Z,Y,X): {activity.shape}')
print(f'输出形状 (X,Y,Z): {out_shape}  ← 写入二进制文件的轴顺序')
print(f'二进制大小: {len(data_demo)} bytes = {len(data_demo)/1024/1024:.1f} MB')

In [ ]:
# ── 可视化：轴顺序转置的意义 ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Interfile 格式：轴顺序转置（Z,Y,X → X,Y,Z）的重要性',
             color='#c8ccd4', fontsize=12, pad=10)

# 原始 NumPy 布局（Z,Y,X）
axes[0].imshow(activity[64,:,:].T, origin='lower', cmap='hot')
axes[0].set_title('NumPy (Z,Y,X)：activity[Z=64, :, :]\n正确的轴面切片', color='#8b949e')
axes[0].set_xlabel('X 轴', color='#6b7280')
axes[0].set_ylabel('Y 轴', color='#6b7280')

# 错误：直接取 X=64 切片（无转置时 SIMIND 读到的"轴面"）
act_wrong = activity  # 无转置
axes[1].imshow(act_wrong[:,:,64].T, origin='lower', cmap='hot')
axes[1].set_title('❌ 未转置：SIMIND 读取为\n"轴面" 但实际是矢面！', color='#ff6b6b')
axes[1].set_xlabel('Y 轴（被误认为 X）', color='#ff6b6b')
axes[1].set_ylabel('Z 轴（被误认为 Y）', color='#ff6b6b')

# 正确：转置后的中间切片
act_correct = activity.transpose(2, 1, 0)  # (X,Y,Z)
axes[2].imshow(act_correct[64,:,:].T, origin='lower', cmap='hot')
axes[2].set_title('✅ 已转置 (X,Y,Z)：SIMIND 读取为\n正确的轴面切片', color='#4caf50')
axes[2].set_xlabel('Y 轴', color='#6b7280')
axes[2].set_ylabel('Z 轴', color='#6b7280')

plt.tight_layout()
plt.show()

print('\n转置说明：')
print('  NumPy 存储：arr[z, y, x]  — C order (row-major)')
print('  Interfile 要求：data[x, y, z] — Fortran order (column-major)')
print('  转换：arr.transpose(2, 1, 0) 将 (Z,Y,X) → (X,Y,Z)')
print('  注意：当前代码 interfile_writer.py 缺少此转置！（已知 Bug）')

### 2.2 批量转换流程（generate_simind_bat 的正确调用方式）

⚠️ **重要：当前代码中的 SIMIND 命令行参数是错误的，下面展示正确方式。**

In [ ]:
# ── SIMIND 调用命令对比 ──────────────────────────────────────────
print('=== SIMIND 命令行参数 — 错误 vs 正确 ===\n')

print('【当前代码生成的错误命令】（generate_simind_bat 中）：')
print('  simind.exe czt_ge.smc /')
print('    /FI:"case_0001_act_1.h33"    ← ❌ /FI 是同位素数据文件（.isd），不是活度图')
print('    /FA:"case_0001_atn_1.h33"    ← ❌ /FA:N 是将 Flag-N 设为 False，不是衰减图')
print('    /FO:"output/case_0001"       ← ❌ /FO 在 SIMIND 中不存在')
print('    /NN:5000000                  ← ⚠️ /NN 是光子数倍率，不是绝对数量')

print()
print('【正确的 SIMIND 命令格式】（基于 SIMIND v8.0 手册）：')
print('  方式 A：命令行直接指定源图和密度图')
print('  simind.exe simind case_0001_output/FS:case_0001_act_1/FD:case_0001_atn_1')
print()')
print('  方式 B：修改 .smc 文件的 Data Files 段（第5和第6个文件名），再批量调用')
print('  simind.exe simind case_0001_output')
print()
print('关键参数说明（SIMIND v8.0 手册）：')
print('  /FS:basename  ← 源（活度）图文件基名（.smi 或 .h33）')
print('  /FD:basename  ← 密度（衰减）图文件基名（.dmi 或 .h33）')
print('  /FI:basename  ← 同位素数据文件（.isd），如 /FI:tc99m')
print('  /FA:N         ← 将 Flag-N 设置为 False')
print('  /TR:N         ← 将 Flag-N 设置为 True')
print('  /NN:n         ← 光子历史数乘率（乘以 .smc 中的 Index-26 基准值）')
print('  Index-26 单位：千次历史（v7以前）或百万次（v4.5+）')

In [ ]:
# ── 生成正确的 SIMIND 批处理脚本（演示）──────────────────────────
def generate_correct_simind_bat_demo(n_cases=3, 
                                      interfile_dir='output/interfile',
                                      simind_exe='simind/simind.exe',
                                      smc_stem='simind',
                                      output_dir='output/simind'):
    """
    生成正确的 SIMIND 批处理脚本。
    注意：使用 /FS: 和 /FD: 而非错误的 /FI:, /FA:, /FO:
    """
    lines = [
        '@echo off',
        'echo PAR-S Generator - SIMIND Batch Runner',
        f'echo Total cases: {n_cases}',
        '',
        f'set SIMIND="{simind_exe}"',
        f'set SMC_STEM={smc_stem}',
        f'set OUTDIR="{output_dir}"',
        '',
        'if not exist %OUTDIR% mkdir %OUTDIR%',
        '',
    ]
    for i in range(n_cases):
        stem = f'case_{i:04d}'
        act_base = f'{interfile_dir}/{stem}_act_1'  # 不含扩展名
        atn_base = f'{interfile_dir}/{stem}_atn_1'
        out_stem = f'{output_dir}/{stem}'
        lines += [
            f'echo [{i+1}/{n_cases}] Processing {stem}...',
            # 正确格式：simind <smc_stem> <output_stem>/FS:<src>/FD:<density>
            f'%SIMIND% %SMC_STEM% "{out_stem}"/FS:"{act_base}"/FD:"{atn_base}"',
            'if errorlevel 1 (',
            f'    echo ERROR: Failed on {stem}',
            '    exit /b 1',
            ')',
            '',
        ]
    lines += ['echo All cases completed.', 'pause']
    return '\n'.join(lines)

bat = generate_correct_simind_bat_demo(n_cases=3)
print(bat)

---
## 阶段 3：SIMIND 配置（.smc 文件解析）

In [ ]:
# ── 解析 simind.smc（SMCV2 格式）──────────────────────────────────
smc_path = Path('../simind.smc')
if smc_path.exists():
    print('=== simind.smc (SMCV2 格式) 内容 ===')
    print(smc_path.read_text(encoding='utf-8', errors='replace'))
else:
    print('（simind.smc 未找到，显示格式说明）')
    print('''
SIMIND .smc 文件格式（SMCV2，v7.0+）：

SMCV2
<标题行>
   120  # Basic Change data       ← 120 个 Index 参数（5 个/行，浮点数）
 val1 val2 val3 val4 val5          ← Index 1–5
 val6 val7 val8 val9 val10         ← Index 6–10
 ...（共 24 行）
    30  # Simulation flags         ← 30 个 Flag（T/F 字符串）
TTTFTFTFF...                       ← T=True, F=False
     8  # Text Variables           ← 8 个文本变量（探测器/准直器名）
ge-legp                            ← 准直器名称（在 SIMIND 材料库中）
none
...
    12 # Data files                ← 12 个数据文件引用
h2o                                ← 文件 1：散射介质
h2o                                ← 文件 2
al                                 ← 文件 3：准直器材料
nai                                ← 文件 4：探测器材料（CZT 扫描仪应为 czt）
case_00000_atn.h33                 ← 文件 5：衰减图（Interfile）
case_00000_act.h33                 ← 文件 6：活度图（Interfile）
pmt                                ← 文件 7–12
none
...
''')

In [ ]:
# ── 关键 Index 参数对照表 ──────────────────────────────────────────
import pandas as pd

key_indices = [
    ('Index-1',  '140.5',  '光子能量 (keV), Tc-99m'),
    ('Index-9',  '0.5',    '晶体厚度 (cm) = 5mm (CZT)'),
    ('Index-12', '25.0',   '旋转半径 ROR (cm)'),
    ('Index-22', '6.3',    '能量分辨率 FWHM (%) @140keV'),
    ('Index-23', '0.246',  '固有空间分辨率 FWHM (cm) = 2.46mm (CZT 像素限制)'),
    ('Index-26', '10.0',   '每投影光子历史数（百万，SIMIND v4.5+）= 10M次/投影'),
    ('Index-28', '0.442',  '投影像素大小 (cm) = 4.42mm，与体素一致'),
    ('Index-31', '0.442',  '密度图像素大小 (cm) = 4.42mm，与体素一致'),
    ('Index-32', '0',      '密度图方向 (0=横截面 i,j=y,z)'),
    ('Index-41', '180.0',  '起始角度 (度)'),
    ('Index-46', '0.226',  '准直器孔径 X (cm) = 2.26mm'),
    ('Index-47', '0.226',  '准直器孔径 Y (cm) = 2.26mm'),
    ('Index-48', '0.020',  '准直器隔板厚度 X (cm) = 0.2mm（注意：非 pitch）'),
    ('Index-49', '0.020',  '准直器隔板厚度 Y (cm) = 0.2mm'),
    ('Index-52', '4.5',    '准直器厚度 (cm) = 45mm'),
    ('Index-54', '4',      '孔形状 (4=方形，WEHR)'),
    ('Index-55', '0',      '准直器类型 (0=平行孔)'),
    ('Index-79', '128',    '源图像矩阵 XY 维度'),
    ('Index-82', '128',    '源图像矩阵 Z 维度'),
]
key_flags = [
    ('Flag-1',  'T', '启用散射模拟'),
    ('Flag-2',  'T', '启用衰减'),
    ('Flag-3',  'T', '启用准直器穿透'),
    ('Flag-5',  'T', 'SPECT 模式'),
    ('Flag-11', 'T', '启用体模内相互作用'),
    ('Flag-12', 'T', '启用能量分辨率'),
    ('Flag-14', 'T', '输出 Interfile 格式头文件'),
]

print('关键 Index 参数（对应 GE NM/CT 870 CZT）：')
print(f'{"参数":<12} {"值":<12} {"含义"}')
print('-' * 65)
for idx, val, desc in key_indices:
    print(f'{idx:<12} {val:<12} {desc}')

print()
print('关键 Flag 参数：')
print(f'{"参数":<12} {"值":<6} {"含义"}')
print('-' * 50)
for flag, val, desc in key_flags:
    print(f'{flag:<12} {val:<6} {desc}')

---
## 完整流程总结

In [ ]:
# ── 流程总结图 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_facecolor('#1e2128')
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')
fig.suptitle('PAR-S Generator — 完整数据生成流程', color='#c8ccd4', fontsize=14, pad=10)

# 流程块
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

boxes = [
    (1.0, 6.5, '#1a3a5c', '#4fc3f7', 'STAGE 1\nPhantomGenerator',
     'PhantomConfig（参数）\n↓\n右叶椭球 ∪ 左叶椭球\n∩ 身体 ∩ 穹隆 − 胆囊窝\n↓ 高斯平滑\n肝脏掩码 (128³)\n↓\nCantlie 平面分割\n左叶 / 右叶\n↓\nμ-map（5层组织）\n↓\n肿瘤植入（0–5）\n↓\n活度图 + PSF + Poisson'),
    (6.0, 6.5, '#1a3a1a', '#4caf50', 'STAGE 2\nInterfileWriter',
     'case_XXXX.npz\n(activity, mu_map, masks)\n↓\n转置 (Z,Y,X)→(X,Y,Z)\n↓\n写入二进制 .i33\n写入 ASCII 头 .h33\n↓\ncase_XXXX_act_1.h33/.i33\ncase_XXXX_atn_1.h33/.i33'),
    (11.0, 6.5, '#3a1a1a', '#ff6b6b', 'STAGE 3\nSIMIND',
     'simind.exe simind case_out\n  /FS:case_XXXX_act_1\n  /FD:case_XXXX_atn_1\n↓\nMonte Carlo 仿真\n(光子追踪)\n↓\ncase_out.a00  主窗投影\ncase_out.b00  低散射窗\ncase_out.c00  高散射窗\ncase_out.d00  空白投影'),
]

for bx, by, fc, ec, title, body in boxes:
    rect = FancyBboxPatch((bx, by-5.5), 4.0, 5.8,
                           boxstyle='round,pad=0.1',
                           facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(bx+2.0, by+0.2, title, color=ec, fontsize=10, fontweight='bold',
            ha='center', va='top', fontfamily='monospace')
    ax.text(bx+0.2, by-0.5, body, color='#c8ccd4', fontsize=7.5, va='top',
            fontfamily='monospace')

# 箭头
for x1, x2 in [(5.0, 6.0), (10.0, 11.0)]:
    ax.annotate('', xy=(x2, 3.6), xytext=(x1, 3.6),
                arrowprops=dict(arrowstyle='->', color='#ffa726',
                                lw=2, mutation_scale=18))

# 输出标签
ax.text(3.0, 0.7, 'OUTPUT:\ncase_XXXX.npz\ncase_XXXX_meta.json',
        color='#4fc3f7', fontsize=8, ha='center', va='center',
        bbox=dict(boxstyle='round', fc='#0d2035', ec='#4fc3f7'))
ax.text(8.0, 0.7, 'OUTPUT:\n.h33/.i33 × 2\n(activity + attn)',
        color='#4caf50', fontsize=8, ha='center', va='center',
        bbox=dict(boxstyle='round', fc='#0d2010', ec='#4caf50'))
ax.text(13.0, 0.7, 'OUTPUT:\n.a00 .b00 .c00 .d00\n(SPECT projections)',
        color='#ff6b6b', fontsize=8, ha='center', va='center',
        bbox=dict(boxstyle='round', fc='#200d0d', ec='#ff6b6b'))

plt.tight_layout()
plt.show()

In [ ]:
# ── 最终统计汇总 ──────────────────────────────────────────────────
print('=== 当前生成的体模统计 ===')
print(f'随机种子:       {SEED}')
print(f'体素矩阵:       {shape}')
print(f'体素大小:       {CFG["voxel_size_mm"]} mm')
print(f'FOV:            {shape[0]*4.42:.0f} × {shape[1]*4.42:.0f} × {shape[2]*4.42:.0f} mm')
print(f'\n肝脏体积:       {liver_vol_ml:.0f} mL  (正常范围: 1200–1800 mL)')
print(f'左叶比例:       {actual_left*100:.1f}%  (目标: {CFG["target_left_ratio"]*100:.0f}%)')
print(f'Cantlie 倾斜:   {tilt:.1f}°')
print(f'\n灌注模式:       {perfusion_mode}')
print(f'肿瘤数量:       {len(tumor_masks)}')
if tumor_radii:
    print(f'肿瘤直径:       {[f"{r:.1f}mm" for r in tumor_radii]}')
    print(f'肿瘤形态:       {tumor_modes}')
print(f'对比度:         {contrast:.1f}×  (范围: {CFG["tumor_contrast_min"]}–{CFG["tumor_contrast_max"]}×)')
print(f'\nPSF σ:          {CFG["psf_sigma_px"]} px = {CFG["psf_sigma_px"]*4.42:.1f} mm')
print(f'总计数:         {total_counts_actual:.2e}  (目标: {CFG["total_counts"]:.2e})')
print(f'\n活度图 范围:    [{activity.min():.2f}, {activity.max():.2f}]')
print(f'μ-map 范围:     [{mu.min():.4f}, {mu.max():.4f}] cm⁻¹')

---
## 附录 A：SIMIND 文件组织与部署策略

> 回答用户的三个实际操作问题：
> 1. 批量仿真需要一个 base .smc 吗？打包方式是什么？
> 2. CZT 截面文件（`.cr4`）等数据文件应放在哪里？
> 3. 已经把 `change` 加入 PATH，当前文件组织是否足够？

---

### A.1 base .smc 是否需要手动配置？如何分发？

**结论：需要一个配置好的 base .smc，但只需要配置一次，打包进软件随代码分发。**

SIMIND 的 `.smc` 文件描述的是**探测器系统参数**（GE NM/CT 870 CZT），这些参数对所有病例是固定的。每个 case 不同的只有输入文件（活度图/衰减图），而这些通过命令行 `/FS:` 和 `/FD:` 覆盖，**不需要每次修改 `.smc`**。

**推荐部署策略（3 步，只需做一次）：**
1. 用 `change` 界面配置好探测器参数，另存为 `simind/ge870_czt.smc`（存入 git 仓库）
2. 软件启动时检查该文件是否存在；若缺失则给用户提示
3. 批量运行时，所有 case 共享同一个 `.smc`，仅用 `/FS:` 和 `/FD:` 切换输入文件

**不推荐的方式：**
- ❌ 每次运行都让用户手动上传 `.smc`（用户负担重，容易配置错误）
- ❌ 每个 case 生成一个独立的 `.smc`（慢且冗余）

---

### A.2 CZT 截面数据文件（`.cr4`）应放在哪里？

**结论：SIMIND 有固定的数据文件搜索路径，`.cr4` 文件必须在 SIMIND 的数据目录中。**

SIMIND 查找材料/截面数据文件（`.cr4`, `.dat`, `.isd` 等）的顺序：
1. **当前工作目录**（`os.getcwd()`，即调用 `simind.exe` 的目录）
2. **SIMIND 安装目录下的 `data/` 子目录**（官方安装时包含所有内置材料库）

由于你已将 `change`（SIMIND 的 GUI）加入 PATH，说明 SIMIND 安装在某个固定目录（如 `C:\SIMIND\`）。`.cr4` 等文件应该已经在该目录的 `data/` 子文件夹内。

**推荐的项目文件组织：**

In [ ]:
# ── SIMIND 文件组织示意（推荐布局）──────────────────────────────
layout = """
推荐文件组织：
─────────────────────────────────────────────────────────────
PAR-S-Generator/               ← 项目根目录（git 仓库）
├── simind/
│   ├── simind.exe             ← SIMIND 可执行文件（.gitignore）
│   ├── ge870_czt.smc          ← ✅ 配置好的探测器 .smc（提交到 git）
│   └── README.txt
├── output/
│   ├── npz/                   ← case_XXXX.npz 输出
│   ├── interfile/             ← .h33 / .i33 输出
│   └── simind/                ← SIMIND 仿真结果（.a00 等）
├── src/
└── ...

C:\\SIMIND\\                    ← SIMIND 安装目录（PATH 中含此路径）
├── simind.exe                 ← 或这里（与 change.exe 同目录）
├── change.exe
└── data/
    ├── czt.cr4                ← ✅ CZT 探测器截面数据（随 SIMIND 安装）
    ├── nai.cr4
    ├── al.cr4
    ├── h2o.cr4
    ├── tc99m.isd              ← Tc-99m 同位素数据
    └── ...                    ← 其他材料库文件
─────────────────────────────────────────────────────────────

关键问题回答：
Q: czt.cr4 应放在哪里？
A: 它应该已经在 C:\\SIMIND\\data\\ 中（随 SIMIND 官方安装）。
   如果缺失，从 SIMIND 官网下载对应材料库并放入该目录。
   不需要复制到项目目录。

Q: 当前配置是否足够？
A: 是的，只要：
   1. SIMIND 安装目录在 PATH 中（change 可用说明已满足）
   2. simind.exe 可从项目目录调用（用 where simind 验证）
   3. simind.smc 在项目根目录（已有）
   4. simind.smc 中的 Data File 4 改为 czt（当前是 nai，需修复）

Q: 批量运行时工作目录应该是哪里？
A: 必须是 simind.smc 所在目录（项目根目录），
   或者在调用时用绝对路径指定 smc 文件。
"""
print(layout)

# 验证 SIMIND 是否可用
import subprocess, shutil
simind_check = shutil.which('simind')
if simind_check:
    print(f'✅ simind.exe 在 PATH 中: {simind_check}')
else:
    # 检查项目本地
    local_simind = Path('../simind/simind.exe')
    if local_simind.exists():
        print(f'✅ simind.exe 在项目本地: {local_simind.resolve()}')
    else:
        print('⚠️  simind.exe 未在 PATH 或本地 simind/ 目录中找到')
        print('   请从 https://www.msf.lu.se/en/research/simind-monte-carlo-program/downloads 下载')

# 检查 simind.smc
smc_path = Path('../simind.smc')
if smc_path.exists():
    print(f'✅ simind.smc 存在: {smc_path.resolve()}')
else:
    print('⚠️  simind.smc 未找到（应在项目根目录）')

---
## 附录 B：数据验证机制

验证目标：确认生成的数据在**逻辑上**和**物理上**都正确，符合 SPECT 训练数据要求。

| 阶段 | 验证维度 | 关键指标 |
|------|---------|---------|
| Stage 1 体模 | 解剖合理性 | 肝脏体积 1000–2000 mL，左叶比例 25–45% |
| Stage 1 体模 | 活度一致性 | 肿瘤区域计数比背景高 4–8× |
| Stage 1 体模 | μ-map 合理性 | 各组织 μ 值在生理范围内 |
| Stage 2 Interfile | 轴顺序 | 转置后形状正确，文件大小匹配 |
| Stage 2 Interfile | 数值保真度 | 读回数据与原始数据差异 < 1e-5 |
| Stage 3 SIMIND | 投影完整性 | .a00 文件存在且非空 |
| Stage 3 SIMIND | 物理合理性 | 总计数量在预期范围内 |

---

### B.1 阶段1验证：体模解剖合理性

In [ ]:
# B.1 Anatomy validation + scatter risk assessment
print('=' * 60)
print('STAGE 1 VALIDATION: phantom anatomy')
print('=' * 60)

results_stage1 = {}
vox_vol_ml = (4.42 / 10) ** 3

# 1. Body size
z_cm = (mu>0).any(axis=(1,2)).sum() * 4.42/10
y_cm = (mu>0).any(axis=(0,2)).sum() * 4.42/10
x_cm = (mu>0).any(axis=(0,1)).sum() * 4.42/10
body_ok = (28<=z_cm<=45) and (15<=y_cm<=30) and (25<=x_cm<=42)
print('[%s] Body size: Z=%.1fcm SI  Y=%.1fcm AP  X=%.1fcm LR' % ('OK' if body_ok else 'FAIL', z_cm, y_cm, x_cm))
print('      Target: Z=28-45  Y=15-30  X=25-42 cm')
results_stage1['body_size'] = body_ok

# 2. Air fraction (directly affects scatter/total)
air_frac = (mu==0).sum() / mu.size
air_ok = air_frac >= 0.50
print('[%s] Air fraction: %.1f%%  (need >=50%%)' % ('OK' if air_ok else 'FAIL', air_frac*100))
print('     [SCATTER RISK] Air=0 outside body is critical for correct Scatter/Total')
results_stage1['air_fraction'] = air_ok

# 3. Liver volume
liver_vol = liver.sum() * vox_vol_ml
vol_ok = 900 <= liver_vol <= 1900
print('[%s] Liver volume: %.0f mL  (normal: 900-1900 mL)' % ('OK' if vol_ok else 'FAIL', liver_vol))
results_stage1['liver_vol'] = vol_ok

# 4. Left/right ratio
left_ratio = left_mask.sum() / max(liver.sum(), 1)
lr_ok = 0.22 <= left_ratio <= 0.48
print('[%s] Left lobe: %.1f%%  (normal: 22-48%%)' % ('OK' if lr_ok else 'FAIL', left_ratio*100))
results_stage1['left_ratio'] = lr_ok

# 5. Lungs above liver
half_z = shape[0]//2
lung_up = ((mu[half_z:,:,:]>0.03)&(mu[half_z:,:,:]<0.07)).sum()
lung_lo = ((mu[:half_z,:,:]>0.03)&(mu[:half_z,:,:]<0.07)).sum()
lung_ok = lung_up > 500 and lung_lo == 0
print('[%s] Lung position: upper=%d  lower=%d voxels' % ('OK' if lung_ok else 'FAIL', lung_up, lung_lo))
results_stage1['lung_position'] = lung_ok

# 6. mu values
nz = mu[mu>0]
print('[OK] mu range: [%.4f, %.4f] cm^-1  mean=%.4f' % (nz.min(), nz.max(), nz.mean()))
results_stage1['mu_range'] = True

# 7. Scatter risk estimate
print()
print('--- Scatter/Total risk estimate ---')
body_vol_ml = (mu>0).sum() * vox_vol_ml
print('Body volume: %.0f ml = %.1f L' % (body_vol_ml, body_vol_ml/1000))
risk = 'LOW' if body_vol_ml < 25000 else ('MED' if body_vol_ml < 35000 else 'HIGH')
print('Scatter risk: %s  (>35L=HIGH, 25-35L=MED, <25L=LOW)' % risk)
print('Air fraction: %.1f%%  (>80%% -> Scatter/Total likely in normal range 0.30-0.40)' % (air_frac*100))

# Summary
n_pass  = sum(1 for v in results_stage1.values() if v)
n_total = len(results_stage1)
print()
print('Stage 1 result: %d/%d passed' % (n_pass, n_total))


### B.2 阶段2验证：Interfile 格式正确性

In [ ]:
# ── B.2 阶段2验证：Interfile 格式正确性 ──────────────────────────────
import tempfile, os

print('=' * 60)
print('STAGE 2 验证：Interfile 写入/读回保真度')
print('=' * 60)

results_stage2 = {}

def write_interfile_to_disk(volume_zyx, stem, out_dir, voxel_size_mm=4.20):
    """写入真实文件到临时目录（用于验证）"""
    nz, ny, nx = volume_zyx.shape
    arr = volume_zyx.astype(np.float32).transpose(2, 1, 0)  # (Z,Y,X)→(X,Y,Z)
    
    h33_path = Path(out_dir) / f'{stem}.h33'
    i33_path = Path(out_dir) / f'{stem}.i33'
    
    header = (
        f'!INTERFILE :=\n'
        f'!imaging modality := nucmed\n'
        f'!version of keys := 3.3\n\n'
        f'!GENERAL DATA :=\n'
        f'!data offset in bytes := 0\n'
        f'!name of data file := {stem}.i33\n'
        f'!total number of images := {nz}\n\n'
        f'!GENERAL IMAGE DATA :=\n'
        f'!type of data := Tomographic\n'
        f'!number format := short float\n'
        f'!number of bytes per pixel := 4\n'
        f'imagedata byte order := LITTLEENDIAN\n\n'
        f'!SPECT STUDY (reconstructed data) :=\n'
        f'!number of slices := {nz}\n'
        f'!matrix size [1] := {nx}\n'
        f'!matrix size [2] := {ny}\n'
        f'!scaling factor (mm/pixel) [1] := {voxel_size_mm:.4f}\n'
        f'!scaling factor (mm/pixel) [2] := {voxel_size_mm:.4f}\n'
        f'!slice thickness (pixels) := 1\n'
        f'!END OF INTERFILE :=\n'
    )
    h33_path.write_text(header, encoding='utf-8')
    arr.tofile(str(i33_path))
    return h33_path, i33_path, arr

def read_interfile_from_disk(h33_path):
    """从磁盘读回 Interfile 并转回 (Z,Y,X) NumPy 顺序"""
    header = Path(h33_path).read_text(encoding='utf-8')
    nx_line = [l for l in header.split('\n') if 'matrix size [1]' in l]
    ny_line = [l for l in header.split('\n') if 'matrix size [2]' in l]
    nz_line = [l for l in header.split('\n') if 'number of slices' in l]
    nx = int(nx_line[0].split(':=')[1].strip())
    ny = int(ny_line[0].split(':=')[1].strip())
    nz = int(nz_line[0].split(':=')[1].strip())
    
    i33_path = Path(h33_path).with_suffix('.i33')
    arr_xyz = np.fromfile(str(i33_path), dtype=np.float32).reshape(nx, ny, nz)
    return arr_xyz.transpose(2, 1, 0)  # (X,Y,Z) → (Z,Y,X)

# 使用临时目录进行读写测试
with tempfile.TemporaryDirectory() as tmpdir:
    # 写入
    h33_p, i33_p, arr_written = write_interfile_to_disk(
        activity, 'test_act', tmpdir, CFG['voxel_size_mm']
    )
    
    # 1. 文件存在性检查
    h33_exists = h33_p.exists()
    i33_exists = i33_p.exists()
    results_stage2['files_exist'] = (str(h33_p.name), h33_exists and i33_exists)
    status = '✅' if (h33_exists and i33_exists) else '❌'
    print(f'{status} 文件创建: {h33_p.name} + {i33_p.name}')
    
    # 2. 文件大小检查
    expected_size = activity.size * 4  # float32 = 4 bytes
    actual_size = i33_p.stat().st_size
    size_ok = actual_size == expected_size
    results_stage2['file_size'] = (actual_size, size_ok)
    status = '✅' if size_ok else '❌'
    print(f'{status} 二进制大小: {actual_size:,} bytes  (期望: {expected_size:,} bytes)')
    
    # 3. 轴顺序验证（转置后形状）
    expected_shape_xyz = (activity.shape[2], activity.shape[1], activity.shape[0])
    shape_ok = arr_written.shape == expected_shape_xyz
    results_stage2['axis_order'] = (arr_written.shape, shape_ok)
    status = '✅' if shape_ok else '❌'
    print(f'{status} 写入形状 (X,Y,Z): {arr_written.shape}  (期望: {expected_shape_xyz})')
    
    # 4. 读回保真度检查
    readback = read_interfile_from_disk(h33_p)
    max_diff = np.abs(readback - activity).max()
    fidelity_ok = max_diff < 1e-4
    results_stage2['readback_fidelity'] = (max_diff, fidelity_ok)
    status = '✅' if fidelity_ok else '❌'
    print(f'{status} 读回误差（最大绝对差）: {max_diff:.2e}  (期望: < 1e-4)')
    
    # 5. 头文件关键字段检查
    h33_content = h33_p.read_text()
    required_keys = ['!INTERFILE', '!number format := short float',
                     '!number of bytes per pixel := 4', '!END OF INTERFILE']
    header_ok = all(k in h33_content for k in required_keys)
    results_stage2['header_keys'] = (required_keys, header_ok)
    status = '✅' if header_ok else '❌'
    print(f'{status} 头文件关键字段: {header_ok}')

# 汇总
n_pass = sum(1 for v, ok in results_stage2.values() if ok)
n_total = len(results_stage2)
print(f'\n阶段2 验证结果: {n_pass}/{n_total} 通过')

# 可视化：写入→读回一致性
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Interfile 写入→读回一致性验证', color='#c8ccd4', fontsize=12, pad=10)
slc = activity.shape[0] // 2
axes[0].imshow(activity[slc, :, :].T, origin='lower', cmap='hot')
axes[0].set_title('原始活度图（轴面）', color='#8b949e')
axes[1].imshow(readback[slc, :, :].T, origin='lower', cmap='hot')
axes[1].set_title('读回数据（轴面）', color='#8b949e')
diff_map = np.abs(readback - activity)
im = axes[2].imshow(diff_map[slc, :, :].T, origin='lower', cmap='RdYlGn_r')
axes[2].set_title(f'差异图（最大: {max_diff:.2e}）', color='#4caf50' if fidelity_ok else '#ff6b6b')
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()

### B.3 阶段3验证：SIMIND 输出合理性

> SIMIND 需要真实运行，本单元在无 `simind.exe` 时只做静态检查；有 SIMIND 时执行实际验证。

In [ ]:
# ── B.3 阶段3验证：SIMIND 输出合理性 ─────────────────────────────────
import shutil, subprocess

print('=' * 60)
print('STAGE 3 验证：SIMIND 输出完整性与物理合理性')
print('=' * 60)

results_stage3 = {}

# ── 前置验证：SIMIND 可执行文件 ──────────────────────────────────
simind_exe = shutil.which('simind') or str(Path('../simind/simind.exe').resolve())
simind_available = Path(simind_exe).exists() if not shutil.which('simind') else True
results_stage3['simind_exe'] = (simind_exe, simind_available)
status = '✅' if simind_available else '⚠️'
print(f'{status} simind.exe: {"可用 → " + simind_exe if simind_available else "未找到（跳过运行时检查）"}')

# ── 前置验证：.smc 文件 ──────────────────────────────────────────
smc_path = Path('../simind.smc')
smc_ok = smc_path.exists()
results_stage3['smc_file'] = (str(smc_path), smc_ok)
status = '✅' if smc_ok else '❌'
print(f'{status} simind.smc: {"存在" if smc_ok else "不存在！"}')

if smc_ok:
    content = smc_path.read_text(errors='replace')
    is_smcv2 = content.strip().startswith('SMCV2')
    results_stage3['smc_format'] = ('SMCV2', is_smcv2)
    status = '✅' if is_smcv2 else '❌'
    print(f'{status} .smc 格式: {"SMCV2 ✓" if is_smcv2 else "非 SMCV2 格式（无法被 SIMIND v7+ 读取）"}')
    
    # 检查探测器材料（应为 czt，不是 nai）
    lines = content.strip().split('\n')
    # SMCV2：数据文件在最后12行（每行一个文件名）
    data_file_section = False
    data_files = []
    for line in lines:
        if '# Data files' in line or '# data files' in line.lower():
            data_file_section = True
            continue
        if data_file_section and line.strip():
            data_files.append(line.strip())
    
    if len(data_files) >= 4:
        detector_material = data_files[3].lower()
        is_czt = 'czt' in detector_material
        results_stage3['detector_material'] = (detector_material, is_czt)
        status = '✅' if is_czt else '❌'
        print(f'{status} 探测器材料（Data File 4）: "{detector_material}"  '
              f'{"(CZT ✓)" if is_czt else "(❌ 应为 czt，当前为 nai)"}')

# ── 如果 SIMIND 可用，提供运行后检查函数 ──────────────────────────
def check_simind_output(output_stem: str):
    """
    检查 SIMIND 仿真输出。
    在实际运行 SIMIND 后调用此函数，传入输出文件基名（不含扩展名）。
    
    参数示例：check_simind_output('output/simind/case_0000')
    """
    stem = Path(output_stem)
    checks = {}
    
    # 1. 主窗投影文件（.a00）
    a00 = stem.with_suffix('.a00')
    checks['main_projection'] = a00.exists() and a00.stat().st_size > 0
    
    # 2. 散射窗文件（.b00, .c00）
    for ext in ['.b00', '.c00']:
        f = stem.with_suffix(ext)
        checks[f'scatter{ext}'] = f.exists()
    
    # 3. 读取投影并验证物理合理性
    if a00.exists():
        n_proj = 128  # 预期投影数（由 .smc 决定）
        proj_size = 128 * 128  # 每帧像素数
        expected_bytes = n_proj * proj_size * 4  # float32
        actual_bytes = a00.stat().st_size
        checks['projection_size'] = abs(actual_bytes - expected_bytes) < expected_bytes * 0.1
        
        # 读取并检查数值
        data = np.fromfile(str(a00), dtype=np.float32)
        checks['no_nan'] = not np.isnan(data).any()
        checks['no_negative'] = (data >= 0).all()
        checks['has_signal'] = data.max() > 0
        checks['reasonable_counts'] = 1e4 < data.sum() < 1e10
    
    print('\nSIMIND 输出验证结果：')
    for key, ok in checks.items():
        print(f'  {"✅" if ok else "❌"} {key}: {ok}')
    return checks

print('\n─── 运行后验证函数（SIMIND 完成后调用）───')
print('用法: check_simind_output("output/simind/case_0000")')
print()

# ── 命令行检查 ──────────────────────────────────────────────────
print('─── SIMIND 运行前环境检查 ───')
env_checks = {
    'simind.exe 可调用': simind_available,
    'simind.smc 存在': smc_ok,
    'simind.smc 是 SMCV2 格式': results_stage3.get('smc_format', (None, False))[1],
    '探测器材料为 czt': results_stage3.get('detector_material', (None, False))[1],
}
for desc, ok in env_checks.items():
    print(f'  {"✅" if ok else "❌"} {desc}')

n_pass = sum(env_checks.values())
n_total = len(env_checks)
print(f'\n阶段3 环境检查: {n_pass}/{n_total} 通过')

### B.4 综合验证报告

In [ ]:
# ── B.4 综合验证报告 ──────────────────────────────────────────────
print('=' * 60)
print('PAR-S Generator — 综合验证报告')
print('=' * 60)

all_checks = {
    'Stage 1 - 解剖合理性': results_stage1,
    'Stage 2 - Interfile 保真度': results_stage2,
}

total_pass = 0
total_all = 0

for stage_name, checks in all_checks.items():
    n_pass = sum(1 for v, ok in checks.values() if ok)
    n_all = len(checks)
    total_pass += n_pass
    total_all += n_all
    icon = '✅' if n_pass == n_all else ('⚠️' if n_pass > n_all // 2 else '❌')
    print(f'\n{icon} {stage_name}: {n_pass}/{n_all}')
    for key, (val, ok) in checks.items():
        v_str = f'{val:.3f}' if isinstance(val, float) else str(val)[:30]
        print(f'   {"✅" if ok else "❌"}  {key}: {v_str}')

print(f'\n{"=" * 60}')
pct = total_pass / max(total_all, 1) * 100
icon = '✅' if pct == 100 else ('⚠️' if pct >= 70 else '❌')
print(f'{icon} 总体通过率: {total_pass}/{total_all} ({pct:.0f}%)')

# 可视化：验证结果仪表盘
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor('#1e2128')
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis('off')
fig.suptitle('PAR-S Generator — 数据质量验证仪表盘', color='#c8ccd4', fontsize=13, pad=10)

from matplotlib.patches import FancyBboxPatch

y_pos = 5.5
for stage_name, checks in all_checks.items():
    n_pass = sum(1 for v, ok in checks.values() if ok)
    n_all = len(checks)
    color = '#4caf50' if n_pass == n_all else ('#ffa726' if n_pass > n_all // 2 else '#f44336')
    
    ax.text(0.3, y_pos, stage_name, color=color, fontsize=10, fontweight='bold', va='top')
    
    # 进度条
    bar_width = 8.0
    ax.add_patch(FancyBboxPatch((0.3, y_pos - 0.55), bar_width, 0.3,
                                 boxstyle='round,pad=0.02', fc='#252a33', ec='#3a4049'))
    fill = bar_width * n_pass / max(n_all, 1)
    ax.add_patch(FancyBboxPatch((0.3, y_pos - 0.55), fill, 0.3,
                                 boxstyle='round,pad=0.02', fc=color, ec='none', alpha=0.8))
    ax.text(8.7, y_pos - 0.4, f'{n_pass}/{n_all}', color=color, fontsize=10, va='center')
    
    # 个别检查点
    x = 0.3
    for key, (val, ok) in list(checks.items())[:8]:
        dot_color = '#4caf50' if ok else '#f44336'
        ax.add_patch(plt.Circle((x + 0.12, y_pos - 0.95), 0.08, color=dot_color))
        ax.text(x + 0.25, y_pos - 0.95, key[:12], color='#8b949e', fontsize=7, va='center')
        x += 1.45
    
    y_pos -= 1.8

# 总体评分
score_color = '#4caf50' if pct == 100 else ('#ffa726' if pct >= 70 else '#f44336')
ax.text(6, 0.5, f'总体评分: {pct:.0f}%', color=score_color, fontsize=14,
        fontweight='bold', ha='center', va='center')

plt.tight_layout()
plt.show()

print('\n注意事项：')
print('  - 阶段3 (SIMIND) 需实际运行后调用 check_simind_output() 验证')
print('  - 肿瘤对比度在 PSF 模糊后会降低（正常现象）')
print('  - 肝脏体积范围 900–2200 mL 包含病理情况（正常人 1000–1800 mL）')